# Overview of the cleaned source tables

What do the cleaned sources contain, and what needs checking before analysis?

**Scope:** VMI funding records cover 2020-2025, alongside the available names and organization-metadata snapshots. The unit is a recipient-year in VMI and a source record in the other tables.
This notebook documents the structure, coverage and data-quality characteristics of the cleaned VMI, names and organization-metadata tables. Its inspection complements the pipeline's automated validation rather than replacing it. The notebook reads existing cleaned tables without creating processed analysis datasets; its cells are designed to execute sequentially in a fresh kernel.

The [Methodology](../METHODOLOGY.md) documents definitions and correction policies. The [Project overview](../README.md) describes the notebook sequence and dataset rebuild process.

A registration-date review later in the notebook reads the processed recipient table and exports a separate diagnostic CSV to `outputs`. It does not overwrite the analysis datasets.

## Cleaned sources and example records

The first previews show example records from each cleaned source. Recipient codes are text identifiers, even when they contain only digits.

The opening cell defines the project path, loads all three CSV files and displays the first five VMI records. Recipient and organization codes are explicitly read as text to preserve their identifier format.

In [ ]:
# pathlib helps to work with file paths in a clean way.
from pathlib import Path

# pandas is the main Python library for working with tables.
import pandas as pd

# This is the main folder of the project.
# Locate the checkout from the project folder or a notebook subfolder.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "analysis_helpers.py").is_file()
     and (p / "config" / "sources.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project folder.")


# Loading the cleaned VMI support data.
# This table contains support recipients, years, request counts, and calculated PMD amounts.
vmi = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "vmi_support_by_recipient.csv", dtype={"recipient_code": "string"})

# Loading the cleaned names data.
# This table contains Lithuanian names and sex classification.
names = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "names.csv")


# Loading the cleaned Rekvizitai data.
# This table contains organization information matched from Rekvizitai.
rekvizitai = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "rekvizitai.csv", dtype={"code": "string"})


# Showing the first 5 rows of the VMI table to confirm the data loaded correctly.
vmi.head()


### Names reference preview

The first five rows show the structure and example values in the cleaned names reference table. These are name records, not funding recipients or organization records. The classification field reflects the reference source rather than independently verified information about any particular person.

In [ ]:
names.head()


### Organization metadata preview

The first five Rekvizitai rows illustrate the available organization details, including identifiers, registration dates and contact or director fields. This is a preview of the metadata source itself, before any linkage to funding records in this notebook. It does not establish that the metadata describes each historical tax year.

In [ ]:
rekvizitai.head()


## Table dimensions and fields

Row counts and column lists describe the inputs available for joining. The three sources have different units of observation, so their row counts do not need to match.
The first output gives each table's size as `(rows, columns)`. The following cell lists the field names separately.

In [ ]:
print("VMI:", vmi.shape)
print("Names:", names.shape)
print("Rekvizitai:", rekvizitai.shape)


### Fields available in each source

The column lists identify the fields available in the three cleaned tables. They distinguish funding measures from names-reference fields and organization metadata, providing context for later transformations and joins. Listing the columns does not itself check whether their values are complete or correct.

In [ ]:
print("VMI columns:", vmi.columns.tolist())
print("Names columns:", names.columns.tolist())
print("Rekvizitai columns:", rekvizitai.columns.tolist())


## Missing-field coverage

These counts show missing values separately for each table. Missing organization metadata can limit later age comparisons; it does not by itself invalidate a funding record.

The first result covers the VMI table. Each value is a count of missing entries in a column, not a count of distinct recipients; the same row may have more than one missing field.

In [ ]:
vmi.isna().sum()


### Missing values in the names reference

The result counts missing values in each names-table column. It describes completeness within this reference source, not the number of organization directors whose names can be matched to it. No values are filled or removed.

In [ ]:
names.isna().sum()


### Missing organization metadata

These counts describe missing fields within the cleaned Rekvizitai table. Missing registration dates can limit age calculations, while missing director or contact fields affect other types of enrichment. These are field-level gaps, not counts of funding recipients that failed to match an organization.

In [ ]:
rekvizitai.isna().sum()


## Observed tax years

The year list establishes the available time window. It does not show whether every recipient appears in every year.

In [ ]:
vmi["tax_year"].sort_values().unique()


## Organization records without director information

The examples below are organization records without a director value.

In [ ]:
rekvizitai[rekvizitai["director"].isna()].head(20)


## Combined data-quality profile

This repeats key checks in one place for the three inputs. The exported validation reports document conflicting recipient-year keys and corrections, as described in the [validation guide](../VALIDATION_GUIDE.md).

A loop prints each table's dimensions and column-level missing counts using the same format. It summarizes the previously shown diagnostics without cleaning, removing or exporting records.

In [ ]:
# Putting the tables into a dictionary so that we can loop through them and print basic quality information.
# without repeating the same code three times.
tables = {
    "vmi": vmi,
    "rekvizitai": rekvizitai,
    "names": names
}

# Loop through each table and print basic quality information.
for table_name, table in tables.items():
    print(table_name)

    # .shape shows the size of the table:
    # first number = rows, second number = columns.
    print("rows, columns:", table.shape)

    # .isna().sum() counts missing values in each column.
    print("missing values:")
    print(table.isna().sum())

    # Print an empty line so the output is easier to read.
    print()


## Supplementary metadata standardization

The following cells work with a temporary copy of the data. They convert registration dates into a consistent format and standardize the gender labels recorded in the source. The original cleaned files remain unchanged. These labels reflect the source data and have not been independently verified.

The first cell creates `rekvizitai_clean`, trims surrounding whitespace from organization codes and converts `start_date` to a pandas date type. Values the parser cannot recognize become missing dates (`NaT`); this does not by itself establish that the original dates are invalid.

In [23]:
# Create a working copy of the Rekvizitai table.
rekvizitai_clean = rekvizitai.copy()

# Convert organization code into text.
rekvizitai_clean["code"] = rekvizitai_clean["code"].astype("string").str.strip()

# Convert start_date into a real date.
rekvizitai_clean["start_date"] = pd.to_datetime(
    rekvizitai_clean["start_date"],
    errors="coerce"
)


### Standardized source gender labels

The temporary organization table receives a separate `leader_gender_clean` column. The three specified Lithuanian source labels are translated to `Male`, `Female` and `Unknown`; other values are left as recorded. The frequency table includes missing values as well as the translated labels. The original `leader_gender` column remains available, and this step does not infer or independently verify anyone's gender.

In [ ]:
# Create a cleaned English version of leader_gender.
rekvizitai_clean["leader_gender_clean"] = rekvizitai_clean["leader_gender"].replace({
    "Vyras": "Male",
    "Moteris": "Female",
    "Nėra duomenų apie lytį": "Unknown"
})

rekvizitai_clean["leader_gender_clean"].value_counts(dropna=False)


### Preview of the temporary metadata table

The first 20 rows show the working copy after identifier trimming, date conversion and label standardization. The preview places the original fields alongside the added cleaned label. These changes exist only in the notebook's working copy and are not written back to the cleaned source file.

In [ ]:
rekvizitai_clean.head(20)


## Supplementary column profiles

This reloads VMI and summarizes each column's type, missingness and example values. Numeric-looking values are a diagnostic only: recipient codes must remain text identifiers.

The opening cell derives the project folder from the current working directory, reloads the cleaned VMI CSV and shows its first five rows. The next cells define and display the column profile.

In [ ]:
from pathlib import Path
import pandas as pd
import re

# If the notebook is inside the notebooks folder, move one level up.
# Otherwise, use the current folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Loading the cleaned VMI table.
vmi = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "vmi_support_by_recipient.csv",
    low_memory=False,
    dtype={"recipient_code": "string"},
)

vmi.head()


### Column-profile calculation

The reusable `profile_columns` function summarizes each column's stored data type, row count, missing values, distinct nonmissing values and numeric-conversion rate. It then creates and displays the VMI column profile.

`numeric_like_percent` is the percentage of all rows whose value can be converted to a number; missing and unconvertible values do not count as successful conversions. When this share is greater than 90%, the profile reports numeric minimum and maximum values. Otherwise, it reports the shortest and longest nonmissing text lengths. Conversion is temporary and does not change the original column. Numeric-looking recipient codes remain identifiers, not quantities.

In [ ]:
# This function creates a summary profile for every column in a table.
def profile_columns(df):
    profiles = []

    for column in df.columns:
        series = df[column]

# Attempts to convert the column's values to numbers
# to assess whether it contains numeric data.
        numeric_version = pd.to_numeric(series, errors="coerce")
        numeric_share = numeric_version.notna().mean()

        profile = {
            "column": column,
            "pandas_type": str(series.dtype),
            "rows": len(series),
            "missing_values": series.isna().sum(),
            "missing_percent": round(series.isna().mean() * 100, 2),
            "unique_values": series.nunique(dropna=True),
            "numeric_like_percent": round(numeric_share * 100, 2),
        }

        # If most values are numeric, show numeric min and max.
        if numeric_share > 0.9:
            profile["lowest_value"] = numeric_version.min()
            profile["highest_value"] = numeric_version.max()
            profile["shortest_text_length"] = None
            profile["longest_text_length"] = None

        # Otherwise, treat the column as text and show shortest/longest lengths.
        else:
            text_series = series.dropna().astype(str)
            profile["lowest_value"] = None
            profile["highest_value"] = None
            profile["shortest_text_length"] = text_series.str.len().min()
            profile["longest_text_length"] = text_series.str.len().max()

        profiles.append(profile)

    return pd.DataFrame(profiles)

vmi_column_profile = profile_columns(vmi)

vmi_column_profile


### Repeated view of the VMI profile

This cell displays the VMI profile again by calling the function defined above. It repeats the same diagnostic on the current table rather than introducing another cleaning step or a different sample.

In [ ]:
profile_columns(vmi)


## Funding records before the recorded registration year

This section loads the processed recipient-year table and identifies records whose registration year is later than their funding year. It reconstructs the negative year difference in `age_before_correction` for review, alongside the existing `organization_age` value. The reconstructed difference is a diagnostic field, not an age used in the analysis.

The output reports both affected recipient-year records and distinct recipient codes, followed by the first 50 records sorted by code and year. Missing or unparseable year values do not meet this comparison and are not included in this particular review table. The table highlights a date inconsistency; it does not determine the correct registration date or alter the project's age correction policy.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Locate the checkout from the project folder or a notebook subfolder.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "analysis_helpers.py").is_file()
     and (p / "config" / "sources.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project folder.")

recipients = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "recipient_analysis.csv",
    dtype={"recipient_code": "string"},
    low_memory=False,
)

tax_year = pd.to_numeric(recipients["tax_year"], errors="coerce")
registration_year = pd.to_numeric(
    recipients["organization_start_year"], errors="coerce"
)

inconsistent_dates = recipients.loc[
    registration_year.gt(tax_year)
].copy()

# Reconstruct the negative age for inspection, without changing source data.
inconsistent_dates["age_before_correction"] = (
    tax_year - registration_year
).loc[inconsistent_dates.index]

review_columns = [
    "recipient_code",
    "recipient_name",
    "tax_year",
    "organization_start_date",
    "organization_start_year",
    "age_before_correction",
    "organization_age",
    "recipient_municipality",
    "legal_form",
    "requests",
    "calculated_pmd_eur",
]

inconsistent_dates = (
    inconsistent_dates[review_columns]
    .sort_values(["recipient_code", "tax_year"])
    .reset_index(drop=True)
)

print(f"Recipient-year records: {len(inconsistent_dates):,}")
print(
    "Unique recipients:",
    f"{inconsistent_dates['recipient_code'].nunique():,}",
)

display(inconsistent_dates.head(50))

### Exported registration-date review table

The complete inconsistency table is written to `outputs/inconsistent_registration_dates.csv`, including records beyond the 50-row preview. The output directory is created if necessary, and an existing file at that path is replaced when the cell runs. UTF-8 encoding with a byte-order mark supports opening Lithuanian text in spreadsheet software. This export is a review artifact; it does not modify the raw, interim or processed source tables.

In [ ]:
review_path = PROJECT_ROOT / "outputs" / "inconsistent_registration_dates.csv"
review_path.parent.mkdir(parents=True, exist_ok=True)

inconsistent_dates.to_csv(
    review_path, index=False, encoding="utf-8-sig"
)

print(f"Full review table saved to: {review_path}")

### Distinct recipients represented in the review

This preview contains up to 20 distinct recipient codes and their names from the inconsistency table. Each code appears once even if several years were flagged. The displayed name comes from the first flagged record in the sorted table, so it is not necessarily the recipient's latest name. Deduplication affects only this preview, not the underlying funding records.

In [ ]:
display(
    inconsistent_dates[
        ["recipient_code", "recipient_name"]
    ].drop_duplicates("recipient_code").head(20)
)

### Funding history for the selected recipient

The recipient code stored in `code_to_review` selects that organization's records from the full processed table. The result shows its available funding years, recorded registration information, analysis age, request counts and calculated funding in chronological order, including years not present in the inconsistency table. A message appears if the selected code has no matching records. The selection is an inspection example, not a correction to the recipient's history.

In [ ]:
code_to_review = "305674792"  # Replace with an actual recipient code.

recipient_history = recipients.loc[
    recipients["recipient_code"].eq(code_to_review),
    [
        "recipient_code",
        "recipient_name",
        "tax_year",
        "organization_start_date",
        "organization_start_year",
        "organization_age",
        "requests",
        "calculated_pmd_eur",
    ],
].sort_values("tax_year").copy()

if recipient_history.empty:
    print("No matching records. Check the recipient code.")
else:
    display(recipient_history)

### Metadata record for the same recipient

The final lookup reads the cleaned Rekvizitai table and displays records matching the same recipient code, after trimming whitespace from the metadata identifier. This provides the metadata context behind the processed history. The lookup is not an independent verification of the registration date, and an empty result means no matching code was found in this cleaned metadata snapshot.

In [ ]:
metadata = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "rekvizitai.csv",
    dtype={"code": "string"},
    low_memory=False,
)

display(
    metadata.loc[metadata["code"].str.strip().eq(code_to_review)]
)

## Reading the results together

Inspection helps identify questions for validation. It does not certify data quality simply because a preview runs without an error.

[Next notebook](01_exploratory_analysis.ipynb) | [Reading guide](../README.md#reading-order)